<img src='images/gdd-logo.png' width='300px' align='right' style="padding: 15px">

# Context Managers

In Python, a context manager is an object that allows you to control the context in which to run code. You can define how the context is created, and then close or "tear down" the context when you are finished.

<a id='resources'></a>
## Managing resources in Python

In any programming language, the usage of resources like files and databases is very common. But it is important to release these resources after usage. Otherwise we can cause memomry issues and risk other unintended side effects.

For example, let's open the following text file and read it in as a string to interact with.

In [159]:
my_file = open('data/example.txt')

In [160]:
text = my_file.read()

In [161]:
print(text)

This is some example text. Yay!



In [162]:
len(text)

32

Notice at the moment, the file is still *open*.

In [163]:
my_file.closed

False

It can cause issues if too many files are open as they take up space in memory.

Uncomment and run the cell below to demonstrate this.

In [164]:
# file_descriptors = []
# for x in range(100000):
#     file_descriptors.append(open('data/example.txt', 'r'))

We get an error message saying that too many files are open. 

*Restart the kernel and continue.*

To avoid situations like above, when we have finished with a file we should close it.

In [165]:
my_file = open('data/example.txt')

In [166]:
my_file.close()

In [167]:
my_file.closed

True

However, it would be very helpful if user have a mechanism for the automatic setup and teardown of resources.

In fact, as the `open()` function is a **context manager**, it can facilitate the proper handling of resources.

The most common way to do so is by using the `with` keyword. As shown below, it allows us to interact with our file by creating creates a *runtime context*, which is then closed afterwards.

In [168]:
with open('data/example.txt') as my_file:
    text = my_file.read()
    length = len(text)
    print(text)
    print(length)

This is some example text. Yay!

32


In [169]:
my_file.closed

True

In [170]:
print(f'The file is {length} characters long and the first word is {text.split()[0]}')

The file is 32 characters long and the first word is This


<a id='caterers'></a>
## Caterers are Context Managers


<img src='images/party.jpeg' width=500px>

Imagine you are hosting a fancy party. You may get caterers to help with the food and refreshments. 

In this situation, the caterers are analogous to the work that context managers do.

|Context Manager|Caterers|
|:---|:---|
|Set up a context|Set up the tables/prepare the food/drinks}
|Run your code|Leave you to party|
|Tear down the context|Clean up the mess|

<a id='use'></a>

## Using a context manager:

To use a context manager you open the context with the keyword `with`. Any code written in the indented block will run in the context.

```python
with <context-manager>(<args>) as <variable-name>:
    # your code here
    # this code is running 'inside the context'
        
# This code runs after the context is removed
```

---

<a id='ex-use'></a>

## <mark>Exercise: Practice using context managers</mark>

For exercise 1 you will need the following information about the data to use:

|File Name|Full Book Name|
|---|---|
|`data/alice.txt`|Alice's Adventures in Wonderland|
|`data/frankenstein.txt`|Frankenstein; or, The Modern Prometheus|
|`data/pride.txt`|Pride and Prejudice|


#### **Exercise 1:** Count how many times Lewis Caroll uses the word rabbit in the first chapter of Alice's Adventures in Wonderland

- Open `"data/alice.txt"` and assign the file to `file`.
- Using `file.read()` assign a new variable text with the contents of `alice.txt`.
- Use the `str.count()` method to count the number of times the word `rabbit` appears.

In [171]:
with open("data/alice.txt") as file:
    text = file.read()

count = text.lower().count("rabbit")
print(f"'rabbit' appears {count} times")

'rabbit' appears 9 times


In [172]:
with open("data/alice.txt") as file:
    text = file.read()

print(file.closed)         # True — file is closed
print(len(text))            # still works — text is independent
print(text[:50])            # still works

True
11321
Alice’s Adventures in Wonderland


CHAPTER I. Down


In [173]:
# %load answers/ex-use1.py
with open('data/alice.txt') as file:
    text = file.read()


num_rabbits = text.lower().split().count('rabbit')

print(f'Lewis Carroll uses the word "rabbit" {num_rabbits} times')


Lewis Carroll uses the word "rabbit" 6 times


## Building your own context managers

There are two ways to build a context manager. With either a `OOP-based` or **`generator-based`** approach. 

### OOP-based context managers

To define custom context managers we need to create a class that implements `__enter__()` and `__exit__()`.

- You can also define `__init__()` to specify arguments that the context manager can take.
- `__exit__()` needs to accept a reference to `self`, the type of exception it might throw, the exception itself and a traceback object as arguments.

In [174]:
import sqlite3
import pandas as pd

class DBConnection:

    def __init__(self, db_name):
        self.db = db_name

    def __enter__(self):
        self.conn = sqlite3.connect(self.db)
        return self.conn

    def __exit__(self, exc_class, exc, traceback):
        self.conn.close()
        
        
with DBConnection('SQLDatabase.db') as db:
     trends = pd.read_sql('''SELECT * FROM programming_trends''', db)

trends.head()

,Month,Python,SQL,R,JavaScript,Visual_Basic_for_Applications
0,2004-01-01,14,84,6,88,14
1,2004-02-01,14,94,6,95,16
2,2004-03-01,15,93,<5,89,15
3,2004-04-01,14,96,7,90,15
4,2004-05-01,12,91,6,87,14


#### **Exercise 2:** Create a context manager called `InDir` that allows you to run code form a different directory than the current working directory.

In [175]:
!pwd

/Users/daria/prod-ready-ml-public/notebooks/context_managers


In [176]:
import os

class InDir:
    """ Temporary change the working directory. """

    def __init__(self, new_dir):
        self.new_dir = new_dir
        self.original_dir = None

    def __enter__(self):
        self.original_dir = os.getcwd()    # remember where we were
        os.chdir(self.new_dir)              # go somewhere new
        return self
    
    def __exit__(self, exc_class, exc, traceback):
        os.chdir(self.original_dir)         # always go back

In [177]:
import os

print(os.getcwd())                # /Users/daria/prod-ready-ml-public/notebooks

with InDir("../"):
    print(os.getcwd())             # /Users/daria/prod-ready-ml-public

print(os.getcwd())                # back to original ✅

/Users/daria/prod-ready-ml-public/notebooks/context_managers
/Users/daria/prod-ready-ml-public/notebooks
/Users/daria/prod-ready-ml-public/notebooks/context_managers


In [178]:
# This code should work unedited
with InDir('../../'):
    notebook_files = os.listdir('notebooks')
    
notebook_files

['decorators_pipelines',
 'oop',
 '02_code_quality.ipynb',
 'context_managers',
 '06_logging.ipynb',
 'fastapi',
 '06_hackathon.ipynb',
 '04_unit_tests.ipynb',
 '05_more_testing.ipynb',
 '01_packaging.ipynb',
 '07_hackathon.ipynb',
 'generators_iterators',
 '03_quality_checks.ipynb']

In [179]:
!pwd # Should not have changed!

/Users/daria/prod-ready-ml-public/notebooks/context_managers


In [180]:
# %load answers/ex-build1.py
import os

class InDir:
    def __init__(self, path):
        self.old_path = os.getcwd()
        self.path = path

    def __enter__(self):
        os.chdir(self.path)
        return None

    def __exit__(self, exc_class, exc, traceback):
        os.chdir(self.old_path)

with InDir('../../'):
    notebook_files = os.listdir('notebooks')

notebook_files


['decorators_pipelines',
 'oop',
 '02_code_quality.ipynb',
 'context_managers',
 '06_logging.ipynb',
 'fastapi',
 '06_hackathon.ipynb',
 '04_unit_tests.ipynb',
 '05_more_testing.ipynb',
 '01_packaging.ipynb',
 '07_hackathon.ipynb',
 'generators_iterators',
 '03_quality_checks.ipynb']

#### **Exercise 3:** Create a context manager called `Timer` that times the execution time of code in its body.

- You can also add an extra argument that allows the user to add a description to the log of the execution time.

In [181]:
import time

class Timer:
    """ Measure the time taken by a block of code. """

    def __init__(self, description="Block of code"):
        self.description = description
        self.start_time = None
        self.end_time = None

    def __enter__(self):
        self.start_time = time.time()
        return self
    
    def __exit__(self, exc_class, exc, traceback):
        self.end_time = time.time()
        print(f'The block of code took {round(self.end_time - self.start_time, 4)} seconds to run.')


with Timer() as t:
    total = 0
    for i in range(1000000):
        total += i

The block of code took 0.038 seconds to run.


In [182]:
with Timer("Heavy computation"):
    sum(i**2 for i in range(10_000_000))

# Heavy computation took 1.2345s

The block of code took 0.4088 seconds to run.


In [183]:
# %load answers/ex-build2.py

### Generator-based context managers

Instead of creating context managers by designing classes, it's usually more ergonomic and idiomatic to use a generator function.

You can decorate any generator with the `@contextlib.contextmanager` decorator. The code before the `yield` statemt will act as the context set-up. The `yield` statement can return a handle to the object created in the context, and any code after `yield` will act as teardowm.

The quivalent to the previous database context manager would be:

In [184]:
import contextlib
import sqlite3
import pandas as pd

@contextlib.contextmanager
def my_database(db_name):
    
    conn = sqlite3.connect(db_name)
    
    yield conn
    
    conn.close()
    

with my_database('SQLDatabase.db') as db:
    trends = pd.read_sql('''SELECT * FROM programming_trends''', db)
    
trends.head()

,Month,Python,SQL,R,JavaScript,Visual_Basic_for_Applications
0,2004-01-01,14,84,6,88,14
1,2004-02-01,14,94,6,95,16
2,2004-03-01,15,93,<5,89,15
3,2004-04-01,14,96,7,90,15
4,2004-05-01,12,91,6,87,14


Python's `sqlite` package actually comes with it's own context manager, great! So we can use that instead. The above demonstrates the flow of using a context manager while connecting to SQL.

In [185]:
import sqlite3

with sqlite3.connect('SQLDatabase.db') as conn:
    query = '''SELECT * FROM programming_trends'''
    results = conn.execute(query).fetchall()

results

[('2004-01-01', 14, 84, 6, 88, 14),
 ('2004-02-01', 14, 94, 6, 95, 16),
 ('2004-03-01', 15, 93, '<5', 89, 15),
 ('2004-04-01', 14, 96, 7, 90, 15),
 ('2004-05-01', 12, 91, 6, 87, 14),
 ('2004-06-01', 14, 100, 6, 89, 17),
 ('2004-07-01', 15, 98, 6, 86, 16),
 ('2004-08-01', 15, 94, '<5', 88, 17),
 ('2004-09-01', 15, 88, 6, 80, 16),
 ('2004-10-01', 15, 89, 7, 77, 14),
 ('2004-11-01', 13, 84, 6, 73, 14),
 ('2004-12-01', 14, 84, 6, 72, 14),
 ('2005-01-01', 14, 82, 7, 69, 14),
 ('2005-02-01', 14, 86, 7, 74, 15),
 ('2005-03-01', 14, 86, 7, 71, 15),
 ('2005-04-01', 13, 85, 7, 72, 14),
 ('2005-05-01', 14, 83, 7, 70, 14),
 ('2005-06-01', 15, 89, 7, 73, 16),
 ('2005-07-01', 14, 82, 7, 68, 14),
 ('2005-08-01', 14, 80, 6, 70, 15),
 ('2005-09-01', 13, 79, 7, 67, 13),
 ('2005-10-01', 15, 76, 7, 66, 13),
 ('2005-11-01', 14, 78, 7, 63, 13),
 ('2005-12-01', 13, 71, 7, 62, 13),
 ('2006-01-01', 13, 71, 7, 59, 12),
 ('2006-02-01', 14, 78, 8, 64, 13),
 ('2006-03-01', 13, 78, 7, 65, 13),
 ('2006-04-01', 13, 7

**Caveat**: Often you would want to include a `try` (`except`) and `finally` within the function to ensure you are able to handle any connection errors you might have.

#### **Exercise 4:** Recreate the context managers from exercises 2 and 3 with generators

In [186]:
# %load answers/ex-build1.py
import os

class InDir:
    def __init__(self, path):
        self.old_path = os.getcwd()
        self.path = path

    def __enter__(self):
        os.chdir(self.path)
        return None

    def __exit__(self, exc_class, exc, traceback):
        os.chdir(self.old_path)

with InDir('../../'):
    notebook_files = os.listdir('notebooks')

notebook_files

['decorators_pipelines',
 'oop',
 '02_code_quality.ipynb',
 'context_managers',
 '06_logging.ipynb',
 'fastapi',
 '06_hackathon.ipynb',
 '04_unit_tests.ipynb',
 '05_more_testing.ipynb',
 '01_packaging.ipynb',
 '07_hackathon.ipynb',
 'generators_iterators',
 '03_quality_checks.ipynb']

In [187]:
import contextlib
import os

@contextlib.contextmanager
def in_dir(new_dir):
    """Temporarily change working directory."""
    original_dir = os.getcwd()
    os.chdir(new_dir)
    try:
        yield                         # no resource to expose, just yield
    finally:
        os.chdir(original_dir)         # always restore, even on errors

In [188]:
import contextlib
import time

@contextlib.contextmanager
def timer(description="Block"):
    """Measure how long the indented code takes."""
    start = time.time()
    try:
        yield
    finally:
        elapsed = time.time() - start
        print(f"{description} took {elapsed:.4f}s")

In [189]:
with in_dir("../"):
    print(os.getcwd())

with timer("Heavy work"):
    sum(i**2 for i in range(10_000_000))

/Users/daria/prod-ready-ml-public/notebooks
Heavy work took 0.4190s


In [190]:
# %load answers/ex-convert2.py
import contextlib
import os
import time

@contextlib.contextmanager
def in_dir(path):
    old_path = os.getcwd()
    os.chdir(path)
    yield
    os.chdir(old_path)

with in_dir('../../'):
    notebook_files = os.listdir('notebooks')

notebook_files

@contextlib.contextmanager
def timer(description):
    start = time.time()
    yield
    end = time.time()
    print(f"{description}: {end - start}")

with timer("timed-time"):
    time.sleep(1.5)


timed-time: 1.519921064376831
